# Tool-Discovery Agent

Discover bioinformatics tools from papers, inject them into your agent, and run tasks.

**Flow:** Scan agent -> Inject tools -> Run tasks via downstream agent

In [ ]:
import os, sys, subprocess
ST2_DIR = '/content/st2'
if os.path.isdir(os.path.join(ST2_DIR, '.git')):
    subprocess.run(['git', '-C', ST2_DIR, 'pull', '--ff-only'], capture_output=True)
else:
    if os.path.exists(ST2_DIR):
        subprocess.run(['rm', '-rf', ST2_DIR], check=True)
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/caixiaoyao2025/st2.git', ST2_DIR], check=True)
ST2_DIR = '/content/st2'
sys.path.insert(0, ST2_DIR)
os.chdir(ST2_DIR)
# Demo input so the default task runs through without the user
# supplying a file. seqmagick is pure-Python/pip-installable, so it
# works in Colab; bqtools would need a Rust/cargo toolchain.
_DEMO_FASTA = ('>seq1 example\n'
              'ACGTACGTACGTACGTACGT\n'
              '>seq2 example\n'
              'TTGGCCTAAGGCCTTAGGCA\n'
              '>seq3 example\n'
              'GATTACAGATTACAGATTACA\n')
with open('reads.fasta', 'w') as _f:
    _f.write(_DEMO_FASTA)
print('demo reads.fasta written:', len(_DEMO_FASTA), 'bytes')
!pip install -q langchain langchain-openai langgraph pydantic graph-tool-call 2>/dev/null || true
from IPython.display import display, Markdown
display(Markdown('**Setup done**'))

## Step 1 - Input your agent & API key

In [ ]:
import os, subprocess
from IPython.display import display, Markdown
import ipywidgets as widgets

# ---- Known agent mappings: method + register method ----
KNOWN_AGENTS = {
    'biomni': {'method': 'go', 'register': 'add_tool'},
    'cellagent': {'method': 'run', 'register': 'add_tool'},
    'geneagent': {'method': 'run', 'register': 'add_tool'},
    'crispr': {'method': 'run', 'register': 'add_tool'},
    'biochatter': {'method': 'run', 'register': 'add_tool'},
    'langchain': {'method': 'invoke', 'register': 'add_tool'},
    'smolagents': {'method': 'run', 'register': 'add_tool'},
    'dspy': {'method': 'forward', 'register': 'add_tool'},
    'crewai': {'method': 'kickoff', 'register': 'add_tool'},
    'metagpt': {'method': 'run', 'register': 'add_tool'},
}
PROBE_ORDER = ['go', 'run', 'execute', 'predict', 'forward', 'invoke']

path_input = widgets.Text(
    value='https://github.com/snap-stanford/Biomni.git',
    placeholder='Git URL or /content/my_agent',
    description='Agent:', layout=widgets.Layout(width='90%'))
key_input = widgets.Password(
    value='sk-CXVEKD43upOkHWTdq1RJP3SMC4OyspQOkqB4ymqw6IJazWyB', placeholder='ark-... or sk-...',
    description='API Key:', layout=widgets.Layout(width='90%'))
model_input = widgets.Text(
    value='deepseek/deepseek-v4-flash-0731',
    description='Model:', layout=widgets.Layout(width='70%'))
base_input = widgets.Text(
    value='https://tokenhub.tencentmaas.com/v1',
    description='Base URL:', layout=widgets.Layout(width='90%'))
btn = widgets.Button(description='Connect agent', button_style='primary')
out = widgets.Output()

def on_connect(_):
    out.clear_output()
    with out:
        agent_src = path_input.value.strip()
        api_key = key_input.value.strip()
        model = model_input.value.strip()
        base_url = base_input.value.strip()
        if not agent_src:
            display(Markdown('**ERROR:** Enter agent path or URL')); return
        if not api_key:
            display(Markdown('**ERROR:** Enter API key')); return
        if agent_src.startswith('http'):
            agent_dir = '/content/_agent_' + agent_src.split('/')[-1].replace('.git', '')
            if not os.path.isdir(agent_dir):
                subprocess.run(['git', 'clone', '--depth', '1', agent_src, agent_dir],
                               capture_output=True, text=True)
            display(Markdown(f'Cloned to `{agent_dir}`'))
        else:
            agent_dir = agent_src
            if not os.path.isdir(agent_dir):
                display(Markdown(f'**ERROR:** `{agent_dir}` not found')); return
        os.environ['OPENAI_API_KEY'] = api_key
        os.environ['OPENAI_BASE_URL'] = base_url
        os.environ['OPENAI_MODEL'] = model
        os.environ['WESTLAKE_API_KEY'] = api_key
        os.environ['BIOMNI_SOURCE'] = 'Custom'
        os.environ['BIOMNI_LLM'] = model
        os.environ['BIOMNI_CUSTOM_BASE_URL'] = base_url
        os.environ['BIOMNI_CUSTOM_API_KEY'] = api_key
        get_ipython().user_ns['agent_dir'] = agent_dir
        get_ipython().user_ns['api_key_val'] = api_key
        get_ipython().user_ns['model_val'] = model
        get_ipython().user_ns['base_url_val'] = base_url
        display(Markdown(f'**Agent:** `{agent_dir}` | **Model:** `{model}` | **Key:** `{api_key[:8]}...`'))

btn.on_click(on_connect)
display(widgets.VBox([path_input, key_input, model_input, base_input, btn, out]))

## Step 2 - Scan agent & detect wiring

In [ ]:
import os, sys, json
from IPython.display import display, Markdown
import yaml

from agent_connector.scanner import build_schema

schema = build_schema(agent_dir, include_evidence=False)
detected = [
    '**Detected:**',
    '- agent_class = `' + str(schema.get('agent_class', 'N/A')) + '`',
    '- module_path = `' + str(schema.get('module_path', 'N/A')) + '`',
    '- registration_method = `' + str(schema.get('registration_method', 'N/A')) + '`',
    '- wiring_style = `' + str(schema.get('wiring_style', 'N/A')) + '`',
    '- init_signature = `' + str(schema.get('init_signature', 'N/A')) + '`',
]
display(Markdown('<br>'.join(detected)))

reg_path = os.path.join(ST2_DIR, 'data', 'mcp_registry.yaml')
tools = yaml.safe_load(open(reg_path, encoding='utf-8'))['tools']
display(Markdown(f'**Registry:** {len(tools)} tools'))

## Step 3 - Resolve execution method

In [ ]:
from IPython.display import display, Markdown

agent_class = (schema.get('agent_class') or '').lower()

# Layer 1: known agent -> direct mapping
exec_method = None
for pat, info in KNOWN_AGENTS.items():
    if pat in agent_class or pat in agent_dir.lower():
        exec_method = info['method']
        display(Markdown(f'**Known agent** `{pat}` -> `{exec_method}`'))
        break

# Layer 2: scan source for .go()/.run() hints
if exec_method is None:
    hits = {}
    for root, dirs, files in os.walk(agent_dir):
        dirs[:] = [d for d in dirs if d not in ('.git','__pycache__','node_modules','.venv')]
        for f in files:
            if f.endswith('.py'):
                try:
                    text = open(os.path.join(root,f), encoding='utf-8', errors='replace').read()
                except: continue
                for m in PROBE_ORDER:
                    hits[m] = hits.get(m, 0) + text.count(f'.{m}(')
    if any(v > 0 for v in hits.values()):
        exec_method = max(hits, key=hits.get)
        display(Markdown(f'**Source hint** -> `{exec_method}` ({hits[exec_method]} occurrences)'))

# Layer 3: default
if exec_method is None:
    exec_method = 'run'
    display(Markdown('**No signal found.** Defaulting to `run`.'))

get_ipython().user_ns['exec_method_override'] = exec_method

## Step 4 - Preflight check & install

In [ ]:
import os, sys, subprocess as _sp
from IPython.display import display, Markdown
from agent_connector.agent_preflight import preflight, preflight_report

# Auto-install missing agent runtime deps (Layer 1) detected by the preflight.
pf = preflight(agent_dir, agent_module_path=schema.get('module_path'), install_missing=True)

# Most OpenAI-compatible agents (incl. BioChatter) also need the LLM backend at
# import time. Layer 2 tool-deps are NOT auto-installed (some are import-tracer
# artifacts like 'base'/'web' that are not real packages), so install the
# known-safe LLM-backend subset explicitly.
#
# For the MCP-native path (BioChatter connects to our server.py as an MCP client)
# we also need langchain-mcp-adapters + nest_asyncio.
for _pkg in ['openai', 'langchain-openai', 'pydantic', 'langchain-community',
             'langchain-mcp-adapters', 'nest_asyncio']:
    try:
        _r = _sp.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg],
                     capture_output=True, text=True, timeout=600)
        if _r.returncode != 0:
            print(f'  (pip install {_pkg} note: {_r.stderr.strip()[:200]})')
    except Exception as _e:
        print(f'  pip install {_pkg} failed: {_e}')

# BioChatter reads its OpenAI-compatible endpoint from GENERIC_TEST_OPENAI_BASE_URL.
_mod = (schema.get('module_path') or '').lower()
if 'biochatter' in agent_dir.lower() or 'biochatter' in _mod:
    os.environ['GENERIC_TEST_OPENAI_BASE_URL'] = os.environ.get('OPENAI_BASE_URL', '')
    display(Markdown('**BioChatter env:** `GENERIC_TEST_OPENAI_BASE_URL` set to the OpenAI base URL.'))

display(Markdown(preflight_report(pf)))


In [ ]:
import subprocess, os, sys
from IPython.display import display, Markdown

if pf.status == 'SETUP_REQUIRED':
    pkgs = pf.pip_installable
    if pkgs:
        display(Markdown(f'Installing **{len(pkgs)}** packages...'))
        cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        display(Markdown(f'**Done** (returncode={r.returncode})'))
    else:
        display(Markdown('No auto-installable packages'))
else:
    display(Markdown(f'Status = `{pf.status}`'))

## Step 5 - Create agent & inject tools

In [ ]:
import os, sys, json, importlib
from IPython.display import display, Markdown

from agent_connector.generator import generate_wiring, load_wrappers, load_adapter

schema['execution_method'] = exec_method_override

import ipywidgets as _widgets
# Agent is auto-detected from the connected repo via build_schema() -- no manual
# selection. Exec mode only chooses how tools are driven: 'Unified (Tool Runner)'
# is agent-agnostic and recommended; 'Native tool calling' lets react/biochatter self-drive,
# otherwise we take over a lightweight tool-calling loop ourselves.
exec_mode = _widgets.Dropdown(options=['Unified (recommended)', 'Native tool calling'],
                              value='Unified (recommended)', description='Exec mode:')
display(exec_mode)


# Force-set BIOMNI env vars BEFORE any agent import (so default_config picks them up)
import os as _os
_m = _os.environ.get('OPENAI_MODEL', '')
_k = _os.environ.get('OPENAI_API_KEY') or _os.environ.get('WESTLAKE_API_KEY')
_u = _os.environ.get('OPENAI_BASE_URL', '')
if _m and not _os.environ.get('BIOMNI_LLM'): _os.environ['BIOMNI_LLM'] = _m
if _k and not _os.environ.get('BIOMNI_CUSTOM_API_KEY'): _os.environ['BIOMNI_CUSTOM_API_KEY'] = _k
if _u and not _os.environ.get('BIOMNI_CUSTOM_BASE_URL'): _os.environ['BIOMNI_CUSTOM_BASE_URL'] = _u
if not _os.environ.get('BIOMNI_SOURCE'): _os.environ['BIOMNI_SOURCE'] = 'Custom'

# Placeholder used only when no native agent can be created, so tool injection
# still works. It is replaced by our lightweight tool-calling loop at the end.
class _LoopHolder:
    def __init__(self): self.tools = []
    def add_tool(self, t): pass

# Generate wiring
wiring_dir = os.path.join(ST2_DIR, 'wiring')
wiring = generate_wiring(tools, schema, out_dir=wiring_dir)
display(Markdown('Wiring mode: `' + wiring['mode'] + '`'))

# Load wrappers as functions (Biomni needs inspect.getsource)
sys.path.insert(0, wiring_dir)
sys.path.insert(0, agent_dir)
wrappers = load_wrappers(package_name='generated_tools', registration_style='function')
display(Markdown(f'**{len(wrappers)} wrappers loaded**'))

# Create agent via adapter
agent = None
adapter = None

if schema.get('module_path') or schema.get('registration_method'):
    try:
        adapter = load_adapter(schema.get('agent_class') or 'Agent',
                               adapter_path=wiring['artifacts']['adapter'])
        agent = adapter.create_agent(use_tool_retriever=False)
        display(Markdown(f'**Agent created via adapter:** `{type(agent).__name__}`'))
    except Exception as e:
        display(Markdown(f'`create_agent()` failed: `{e}`'))

# Fallback: if adapter.create_agent() failed (e.g. react needs all Biomni deps), try A1
# A1 is lighter -- it doesn't load all built-in tools in __init__
if agent is None and 'biomni' in agent_dir.lower():
    try:
        sys.path.insert(0, agent_dir)
        from biomni.agent.a1 import A1
        agent = A1(
            path=os.path.join(ST2_DIR, '_agent_data'),
            llm=os.environ.get('OPENAI_MODEL', 'gpt-4o'),
            source='Custom',
            base_url=os.environ.get('OPENAI_BASE_URL'),
            api_key=os.environ.get('OPENAI_API_KEY'),
            use_tool_retriever=False,
            expected_data_lake_files=[],
        )
        display(Markdown(f'**A1 agent created:** `{type(agent).__name__}`'))
    except Exception as e:
        display(Markdown(f'A1 fallback failed: `{e}`'))

# If still no native agent, fall back to the placeholder (replaced below).
if agent is None:
    agent = _LoopHolder()
    display(Markdown('**No native agent created** -- will drive tools with our lightweight tool-calling loop.'))

# Inject tools: create LangChain tools from _TOOL_SPEC and append to agent.tools
from langchain_core.tools import tool as lc_tool
from agent_connector.tool_runner import run_tool_spec, format_result

# Deterministic schema generation for A1's add_tool: building the API
# schema via the LLM is flaky (truncation / 'not applicable') AND burns the
# LLM quota (402 free-trial exhaustion). Our wrappers carry a _TOOL_SPEC,
# so build the schema directly from it -- no LLM call.
try:
    import ast as _ast
    import biomni.agent.a1 as _a1mod
    _orig_ftas = _a1mod.function_to_api_schema
    _PATCH_A1 = 'a1' in (schema.get('agent_class') or '').lower()
    if not _PATCH_A1:
        raise RuntimeError('skip A1 monkeypatch for non-A1 agent')

    _SPEC_BY_NAME = {}
    try:
        import sys as _sys
        for _w in wrappers:
            _n = getattr(_w, "__name__", None)
            _mod = _sys.modules.get(getattr(_w, "__module__", "")) if _n else None
            _s = getattr(_mod, "_TOOL_SPEC", None) if _mod else None
            if _s is None:
                _s = getattr(_w, "_TOOL_SPEC", None)
            if _n and _s:
                _SPEC_BY_NAME[_n] = _s
                _sn = _s.get("name")
                if _sn:
                    _SPEC_BY_NAME[_sn] = _s
    except Exception:
        pass

    def _reliable_function_to_api_schema(function_string, llm):
        _spec = None
        try:
            _tree = _ast.parse(function_string)
            for _node in _tree.body:
                if isinstance(_node, _ast.Assign):
                    for _t in _node.targets:
                        if isinstance(_t, _ast.Name) and _t.id == '_TOOL_SPEC':
                            _spec = _ast.literal_eval(_node.value)
                elif isinstance(_node, (_ast.FunctionDef, _ast.ClassDef)) and _spec is None:
                    _spec = _SPEC_BY_NAME.get(_node.name)
        except Exception:
            _spec = None
        if _spec is None:
            try:
                _t2 = _ast.parse(function_string)
                for _node in _t2.body:
                    if isinstance(_node, (_ast.FunctionDef, _ast.ClassDef)):
                        _spec = _SPEC_BY_NAME.get(_node.name)
                        if _spec:
                            break
            except Exception:
                _spec = None
        if _spec is not None:
            _req, _opt = [], []
            for _pname, _meta in (_spec.get('inputs') or {}).items():
                if _pname == 'subcommand':
                    continue
                _m = _meta or {}
                _entry = {'name': _pname, 'type': 'str',
                          'description': _m.get('description', ''), 'default': None}
                (_req if _m.get('required') else _opt).append(_entry)
            return {'name': _spec.get('name'),
                    'description': _spec.get('description', ''),
                    'required_parameters': _req,
                    'optional_parameters': _opt}
        return _orig_ftas(function_string, llm)

    _a1mod.function_to_api_schema = _reliable_function_to_api_schema
except Exception:
    pass

if not hasattr(agent, 'tools') or agent.tools is None:
    agent.tools = []

injected = 0
_register_via_add_tool = hasattr(agent, 'add_tool')
for w in wrappers:
    ts = getattr(w, '_TOOL_SPEC', None)
    if ts is None and hasattr(w, '__module__'):
        mod = importlib.import_module(w.__module__)
        ts = getattr(mod, '_TOOL_SPEC', None)
    if ts:
        def _make_runner(spec=ts):
            _desc = spec.get('description', spec['name'])
            _inputs = spec.get('inputs') or {}
            if _inputs:
                _params = []
                for _pname, _pmeta in _inputs.items():
                    _req = 'required' if (_pmeta or {}).get('required') else 'optional'
                    _ptype = (_pmeta or {}).get('type', 'string')
                    _pdesc = (_pmeta or {}).get('description', '')
                    _params.append(f'  {_pname} ({_ptype}, {_req}): {_pdesc}')
                _desc += '\nParameters:\n' + chr(10).join(_params)
            def fn(**kwargs):
                return format_result(run_tool_spec(spec, dict(kwargs)))
            fn.__name__ = spec['name']
            fn.__doc__ = _desc
            return lc_tool(fn)
        agent.tools.append(_make_runner())
        injected += 1
    else:
        agent.tools.append(w)
        injected += 1
    if _register_via_add_tool:
        try:
            agent.add_tool(w)
        except Exception as _e:
            display(Markdown(f'`add_tool({getattr(w, "__name__", "tool")})` failed: `{_e}`'))

display(Markdown(f'**{injected} tools injected** into `{type(agent).__name__}.tools`'))

tool_names = []
for t in agent.tools:
    name = getattr(t, 'name', None) or getattr(t, '__name__', '?')
    tool_names.append(name)
display(Markdown('**Available tools:** ' + ', '.join(tool_names)))

is_known = any(pat in agent_dir.lower() or pat in type(agent).__name__.lower()
               for pat in KNOWN_AGENTS)
if is_known:
    display(Markdown(f'**Known agent** -> method `{exec_method_override}` (trusted)'))
else:
    probe = None
    for m in PROBE_ORDER:
        if m == '__call__':
            if callable(agent): probe = m; break
        elif hasattr(agent, m) and callable(getattr(agent, m)):
            probe = m; break
    if probe:
        display(Markdown(f'**Probe OK:** `agent.{probe}()` exists'))
        get_ipython().user_ns['exec_method_override'] = probe
    else:
        tried = ', '.join(f'`{m}()`' for m in PROBE_ORDER)
        display(Markdown(f'**Probe FAILED:** tried {tried}'))
        display(Markdown('Go to **Step 5b** to provide your agent init code manually.'))

get_ipython().user_ns['agent'] = agent
get_ipython().user_ns['wrappers'] = wrappers
get_ipython().user_ns['adapter'] = adapter

# Detect agent type: tool calling vs code execution
_name_lower = type(agent).__name__.lower()
_use_native = (exec_mode.value == 'Native tool calling')
_is_tool_calling = _use_native and (
    'react' in _name_lower or
    'biochatter' in _name_lower or
    hasattr(agent, 'tool_executor')
)
if _use_native and not _is_tool_calling:
    display(Markdown('**Note:** native tool calling unavailable for this agent; '
                     'we will drive tools with our lightweight tool-calling loop.'))
display(Markdown(f'**Exec mode:** `{"Native" if _is_tool_calling else "Unified (Tool Runner)"}` '
                 f'for agent `{type(agent).__name__}`'))

if _is_tool_calling and agent.tools:
    _tool_retriever = None
    display(Markdown(f'**Tool calling agent** -> {len(agent.tools)} tools ready'))
else:
    from agent_connector.graph_retrieval import build_graph_from_tools, retrieve_tools
    from agent_connector.artifact_spec import annotate_inputs_with_artifacts as _annotate_inputs_with_artifacts
    import yaml as _yaml
    _reg_path = os.path.join('data', 'mcp_registry.yaml')
    if os.path.exists(_reg_path):
        with open(_reg_path, 'r', encoding='utf-8') as _f:
            _reg = _yaml.safe_load(_f) or {}
        _tool_graph = build_graph_from_tools(_reg.get('tools', []))
        _injected_dicts = []
        for t in wrappers:
            ts = getattr(t, '_TOOL_SPEC', None)
            if ts:
                _injected_dicts.append(ts)
        if _injected_dicts:
            _injected_graph = build_graph_from_tools(_injected_dicts)
        else:
            _injected_graph = None
        _spec_lookup = {}
        for t in _reg.get('tools', []):
            _spec_lookup[t['name']] = t
            if t.get('subcommand_details'):
                for sub, detail in t['subcommand_details'].items():
                    _fname = f"{t['name']}_{sub.replace('-', '_')}"
                    _leaf = dict(t)
                    _leaf['name'] = _fname
                    _leaf['description'] = detail.get('description', '')
                    _leaf['inputs'] = _annotate_inputs_with_artifacts(
                        {p['name'].lstrip('-').replace('-', '_'): dict(p)
                         for p in (detail.get('params') or [])},
                        tool_name=t['name'], sub_name=sub)
                    _leaf['_active_subcommand'] = sub
                    if not _leaf.get('execution'):
                        _leaf['execution'] = {'type': _leaf.get('type', 'cli'),
                                              'command': _leaf.get('command', '')}
                    _spec_lookup[_fname] = _leaf
        for t in wrappers:
            ts = getattr(t, '_TOOL_SPEC', None)
            if ts:
                _spec_lookup[ts['name']] = ts
        _tool_retriever = {
            'graph': _tool_graph,
            'injected_graph': _injected_graph,
            'spec_lookup': _spec_lookup,
        }
        _enc = _spec_lookup.get('bqtools_encode')
        if _enc and 'input' in (_enc.get('inputs') or {}):
            _ei = _enc['inputs']['input']
            _ei['artifact_type'] = 'fasta'
            _ei['extensions'] = ['.fa', '.fasta', '.fq', '.fastq']
            _ei['required'] = True
        _dec = _spec_lookup.get('bqtools_decode')
        if _dec and 'input' in (_dec.get('inputs') or {}):
            _di = _dec['inputs']['input']
            _di['artifact_type'] = 'binseq'
            _di['extensions'] = ['.vbq', '.cbq', '.bq']
        _rc = _spec_lookup.get('bqtools_revcomp')
        if _rc and 'input' in (_rc.get('inputs') or {}):
            _ri = _rc['inputs']['input']
            _ri['artifact_type'] = 'binseq'
            _ri['extensions'] = ['.vbq', '.cbq', '.bq']
        display(Markdown(f'**Code execution agent** -> retriever built ({len(_spec_lookup)} tools indexed)'))
    else:
        _tool_retriever = None
        display(Markdown('**Warning:** mcp_registry.yaml not found, no retrieval'))

# Always expose an OpenAI-compatible client for the lightweight loop.
_key = get_ipython().user_ns.get('api_key_val') or os.environ.get('OPENAI_API_KEY') or ''
_base = get_ipython().user_ns.get('base_url_val') or os.environ.get('OPENAI_BASE_URL') or ''
_model = get_ipython().user_ns.get('model_val') or os.environ.get('OPENAI_MODEL') or 'deepseek/deepseek-v4-flash-0731'
from openai import OpenAI as _OAI
_oai = _OAI(api_key=_key, base_url=_base)
get_ipython().user_ns['_oai_client'] = _oai
get_ipython().user_ns['_oai_model'] = _model

# If no native agent was created, drive the tools ourselves. For agents that
# speak MCP (e.g. BioChatter) we expose our registry through the repo's own
# FastMCP server (server.py) and let the agent connect as an MCP client -- this
# is genuine native tool calling, not our loop. Otherwise we fall back to our own
# lightweight tool-calling loop.
if isinstance(agent, _LoopHolder):
    _is_biochatter = ('biochatter' in agent_dir.lower()) or (
        'biochatter' in (schema.get('module_path') or '').lower())
    if _use_native and _is_biochatter:
        from agent_connector.mcp_bridge import MCPToolBridge, default_server_params
        agent = MCPToolBridge(server_params=default_server_params(),
                              model=_model, base_url=_base, api_key=_key, max_iter=6)
        agent.tools = wrappers
        _is_tool_calling = True
        exec_method_override = 'run'
        display(Markdown('**BioChatter-native via MCP:** our server.py tools are exposed to the agent as an MCP client.'))
    else:
        from agent_connector.lightweight_loop import LightweightToolLoop
        agent = LightweightToolLoop(tool_retriever=_tool_retriever, openai_client=_oai, model=_model)
        agent.tools = wrappers
        _is_tool_calling = True
        exec_method_override = 'run'
        display(Markdown('**Lightweight tool-calling loop ready** (we drive the tools).'))

get_ipython().user_ns['_tool_retriever'] = _tool_retriever
get_ipython().user_ns['_is_tool_calling'] = _is_tool_calling
get_ipython().user_ns['exec_method_override'] = exec_method_override


## Step 5b - Manual agent init (only if Step 5 failed)

If Step 5 succeeded, **skip this cell**.

In [ ]:
from IPython.display import display, Markdown
import ipywidgets as widgets

already_ok = is_known
if not already_ok:
    for m in PROBE_ORDER:
        if m == '__call__':
            if callable(agent): already_ok = True; break
        elif hasattr(agent, m) and callable(getattr(agent, m)):
            already_ok = True; break

if already_ok:
    display(Markdown('Step 5 succeeded. **Skipping.**'))
else:
    display(Markdown('### Agent execution method not recognized\n\n'
        '**Tried:** ' + ', '.join(f'`{m}`' for m in PROBE_ORDER) + '\n\n'
        '**Please paste your agent init code below:**'))
    code_area = widgets.Textarea(
        value='# from my_agent import MyAgent\n# agent = MyAgent(model="gpt-4")\n# agent.add_tools(wrappers)\n',
        placeholder='agent = MyAgent(...)',
        layout=widgets.Layout(width='90%', height='150px'))
    method_input = widgets.Dropdown(
        options=['run', 'execute', 'go', 'predict', 'forward', 'invoke', '__call__'],
        value='run', description='Exec method:', layout=widgets.Layout(width='60%'))
    apply_btn = widgets.Button(description='Apply', button_style='primary')
    out = widgets.Output()

    def on_apply(_):
        out.clear_output()
        with out:
            try:
                local_ns = {'wrappers': wrappers, 'agent_dir': agent_dir}
                exec(code_area.value, local_ns)
                new_agent = local_ns.get('agent')
                if new_agent is None:
                    display(Markdown('Error: no `agent` variable found.')); return
                method = method_input.value
                fn = getattr(new_agent, method, None) if method != '__call__' else new_agent
                if fn is None or not callable(fn):
                    display(Markdown(f'Error: `{method}` not callable.')); return
                get_ipython().user_ns['agent'] = new_agent
                get_ipython().user_ns['exec_method_override'] = method
                agent = new_agent
                display(Markdown(f'**Done!** Agent=`{type(new_agent).__name__}` method=`{method}`'))
            except Exception as e:
                display(Markdown(f'Error: `{e}`'))

    apply_btn.on_click(on_apply)
    display(widgets.VBox([code_area, method_input, apply_btn, out]))

## Step 6 - Run tasks via downstream agent

Your query is sent to the agent via `agent.{method}(query)`. The agent uses the injected tools internally.

**Default task:** summarize a FASTA file - this uses `seqmagick_info` (pure-Python, pip-installable, no Rust needed) and runs out of the box in Colab. `bqtools` BINSEQ tools are also discovered, but they require a Rust/`cargo` toolchain; if selected, the runner reports them as unavailable with their install contract instead of failing silently.

In [ ]:
import os, re as _re, time
from IPython.display import display, Markdown
import ipywidgets as widgets

method = exec_method_override
if method == '__call__':
    exec_fn = agent
else:
    exec_fn = getattr(agent, method, None)

if exec_fn is None or not callable(exec_fn):
    display(Markdown(f'ERROR: `agent.{method}()` not callable'))
else:
    display(Markdown(f'Ready: `agent.{method}()`'))

if _is_tool_calling and agent.tools:
    display(Markdown('**Mode:** Tool calling (agent.tools)'))
else:
    display(Markdown('**Mode:** Retrieval -> Prompt -> Code execution (lightweight loop)'))

query_input = widgets.Text(
    value='I have a FASTA file reads.fasta. Summarize reads.fasta: report the number of sequences and the total length of all sequences.',
    placeholder='Ask a bioinformatics question...',
    description='Query:', layout=widgets.Layout(width='95%'))
run_btn = widgets.Button(description='Run agent', button_style='success')
result_out = widgets.Output()

def run_query(_):
    result_out.clear_output()
    with result_out:
        query = query_input.value.strip()
        if not query:
            display(Markdown('**Enter a query**')); return
        display(Markdown(f'**Query:** {query}'))
        t0 = time.time()
        try:
            if _is_tool_calling and agent.tools:
                display(Markdown(f'**Calling:** `agent.{method}(query)` ...'))
                final = exec_fn(query)
            else:
                _oai = get_ipython().user_ns.get('_oai_client')
                _model = get_ipython().user_ns.get('_oai_model') or os.environ.get('OPENAI_MODEL') or 'deepseek/deepseek-v4-flash-0731'
                from agent_connector.lightweight_loop import LightweightToolLoop
                loop = LightweightToolLoop(tool_retriever=_tool_retriever,
                                           openai_client=_oai, model=_model)
                display(Markdown('**Running lightweight tool-calling loop...**'))
                final = loop.run(query)
            elapsed = time.time() - t0
            display(Markdown(f'**Done** ({elapsed:.1f}s)'))
            display(Markdown(f'### Result\n{str(final)[:3000]}'))
        except Exception as e:
            display(Markdown(f'**Error:** `{type(e).__name__}: {e}`'))

run_btn.on_click(run_query)
display(widgets.VBox([query_input, run_btn, result_out]))
